# 02. Preprocessing the 3W Dataset

The preprocessing stage converts the raw 3W time-series recordings into a consistent representation suitable for feature engineering and machine learning.

The dataset contains 2,228 independent recordings with different durations, sensor availability, missing values, operating conditions, and event-label sequences. Therefore, preprocessing must preserve the temporal structure of each recording while ensuring that the resulting data is consistent across instances.

The main objectives of this stage are:

- Establish a consistent structure for processing all recordings.
- Identify and handle missing sensor observations.
- Validate timestamp ordering and sampling intervals.
- Handle `class` and `state` labels appropriately.
- Preserve the separation between individual recordings.
- Avoid data leakage between recordings during later model development.
- Produce a clean, reproducible dataset for feature engineering.

Because the complete dataset contains more than 76 million observations, recordings will be processed individually rather than loading the entire dataset into memory at once.

In [1]:
# Import libraries required for preprocessing

from pathlib import Path
import pandas as pd
import numpy as np

# Define the project and dataset paths
project_path = Path.cwd().parent
dataset_path = project_path / "data" / "raw" / "3W" / "dataset"

# Display the paths being used
print("Project path:", project_path)
print("Dataset path:", dataset_path)

# Verify that the dataset directory exists
print("Dataset exists:", dataset_path.exists())

Project path: c:\Users\Dell\Desktop\Calibrated-Cost-Sensitive-Rare-Event-Detection
Dataset path: c:\Users\Dell\Desktop\Calibrated-Cost-Sensitive-Rare-Event-Detection\data\raw\3W\dataset
Dataset exists: True


### Identifying the Raw Recordings

Each Parquet file represents an independent time-series recording. The recordings are distributed across folders `0` through `9`, where each folder corresponds to a specific operating condition or event category.

The preprocessing pipeline will treat each Parquet file as an independent recording. This preserves the boundaries between recordings and prevents unrelated time-series segments from being combined.

In [2]:
# Collect all Parquet recordings from the dataset folders

parquet_files = sorted(dataset_path.glob("*/*.parquet"))

# Count the total number of recordings
print("Total Parquet recordings:", len(parquet_files))

# Display the first few file paths
print("\nFirst 5 recordings:")
for file_path in parquet_files[:5]:
    print(file_path)

Total Parquet recordings: 2228

First 5 recordings:
c:\Users\Dell\Desktop\Calibrated-Cost-Sensitive-Rare-Event-Detection\data\raw\3W\dataset\0\WELL-00001_20170201010207.parquet
c:\Users\Dell\Desktop\Calibrated-Cost-Sensitive-Rare-Event-Detection\data\raw\3W\dataset\0\WELL-00001_20170201060114.parquet
c:\Users\Dell\Desktop\Calibrated-Cost-Sensitive-Rare-Event-Detection\data\raw\3W\dataset\0\WELL-00001_20170201110124.parquet
c:\Users\Dell\Desktop\Calibrated-Cost-Sensitive-Rare-Event-Detection\data\raw\3W\dataset\0\WELL-00001_20170201160311.parquet
c:\Users\Dell\Desktop\Calibrated-Cost-Sensitive-Rare-Event-Detection\data\raw\3W\dataset\0\WELL-00001_20170201210228.parquet


### Creating a Recording-Level Inventory

Before modifying any sensor values, a lightweight inventory of the recordings is created.

For each recording, the folder number identifies its event category, while the filename identifies the individual recording. The inventory allows the preprocessing pipeline to track each recording independently throughout subsequent steps.

Only file-level information is collected at this stage; the actual Parquet data is not loaded into memory.

In [3]:
# Create a lightweight inventory of all recordings

recording_inventory = []

for file_path in parquet_files:
    # The parent folder represents the dataset class/category
    folder_number = file_path.parent.name

    # Store basic information without loading the recording
    recording_inventory.append(
        {
            "recording_id": file_path.stem,
            "folder": int(folder_number),
            "file_path": file_path
        }
    )

# Convert the inventory into a DataFrame
recording_inventory = pd.DataFrame(recording_inventory)

# Display the first few recordings
recording_inventory.head()

,recording_id,folder,file_path
0,WELL-00001_20170201010207,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...
1,WELL-00001_20170201060114,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...
2,WELL-00001_20170201110124,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...
3,WELL-00001_20170201160311,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...
4,WELL-00001_20170201210228,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...


### Inspecting Recording Metadata

The recording inventory is extended with the number of observations and the start and end timestamps of each recording.

This metadata is useful for identifying differences in recording length and temporal coverage before applying any transformations. The Parquet files are still processed individually, so the complete dataset is never loaded into memory at once.

In [4]:
# Add basic time-series metadata for each recording

observation_counts = []
start_times = []
end_times = []

for file_path in recording_inventory["file_path"]:
    # Load one recording at a time
    recording = pd.read_parquet(file_path)

    # Store the number of observations
    observation_counts.append(len(recording))

    # Store the first and last timestamps
    start_times.append(recording.index.min())
    end_times.append(recording.index.max())

# Add the metadata to the inventory
recording_inventory["observations"] = observation_counts
recording_inventory["start_time"] = start_times
recording_inventory["end_time"] = end_times

# Display the updated inventory
recording_inventory.head()

,recording_id,folder,file_path,observations,start_time,end_time
0,WELL-00001_20170201010207,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...,21474,2017-02-01 01:02:07,2017-02-01 07:00:00
1,WELL-00001_20170201060114,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...,21527,2017-02-01 06:01:14,2017-02-01 12:00:00
2,WELL-00001_20170201110124,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...,21517,2017-02-01 11:01:24,2017-02-01 17:00:00
3,WELL-00001_20170201160311,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...,21410,2017-02-01 16:03:11,2017-02-01 22:00:00
4,WELL-00001_20170201210228,0,c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...,21453,2017-02-01 21:02:28,2017-02-02 03:00:00


### Checking Recording Duration

Recording duration is calculated from the start and end timestamps. This provides a consistent measure of the temporal length of each recording and helps determine the range of time-series lengths that the preprocessing pipeline must accommodate.

The duration is calculated without assuming a fixed sampling frequency or fixed number of observations.

In [5]:
# Calculate the duration of each recording

recording_inventory["duration"] = (
    recording_inventory["end_time"] - recording_inventory["start_time"]
)

# Display summary statistics for recording duration
print(recording_inventory["duration"].describe())

# Display the shortest and longest recordings
print("\nShortest recording:")
print(recording_inventory.loc[recording_inventory["duration"].idxmin()])

print("\nLongest recording:")
print(recording_inventory.loc[recording_inventory["duration"].idxmax()])

count                         2228
mean     0 days 09:32:53.918312387
std      0 days 13:57:18.524840706
min                0 days 01:26:54
25%         0 days 05:55:41.750000
50%                0 days 07:29:58
75%                0 days 08:08:18
max                8 days 21:25:13
Name: duration, dtype: object

Shortest recording:
recording_id                            WELL-00016_20180426131710
folder                                                          5
file_path       c:\Users\Dell\Desktop\Calibrated-Cost-Sensitiv...
observations                                                 5215
start_time                                    2018-04-26 13:17:10
end_time                                      2018-04-26 14:44:04
duration                                          0 days 01:26:54
Name: 1650, dtype: object

Longest recording:
recording_id                            WELL-00022_20180925003447
folder                                                          7
file_path       c:\Users\Dell

### Defining the Sensor Variables

The raw recordings contain 27 sensor and operational variables in addition to the `class` and `state` labels.

The preprocessing pipeline should distinguish between sensor variables and target-related columns. Sensor variables will be treated as model inputs, while `class` will be retained as the event label and `state` will be preserved for analysis and later validation.

The timestamp index is also retained because the temporal order of observations is essential for time-series processing.

In [6]:
# Load one representative recording to identify the raw column structure

sample_recording = pd.read_parquet(parquet_files[0])

# Separate sensor variables from label columns
label_columns = ["class", "state"]

sensor_columns = [
    column
    for column in sample_recording.columns
    if column not in label_columns
]

# Display the identified columns
print("Number of sensor variables:", len(sensor_columns))
print("\nSensor variables:")
for column in sensor_columns:
    print("-", column)

print("\nLabel columns:", label_columns)
print("Timestamp stored as index:", sample_recording.index.name)

Number of sensor variables: 27

Sensor variables:
- ABER-CKGL
- ABER-CKP
- ESTADO-DHSV
- ESTADO-M1
- ESTADO-M2
- ESTADO-PXO
- ESTADO-SDV-GL
- ESTADO-SDV-P
- ESTADO-W1
- ESTADO-W2
- ESTADO-XO
- P-ANULAR
- P-JUS-BS
- P-JUS-CKGL
- P-JUS-CKP
- P-MON-CKGL
- P-MON-CKP
- P-MON-SDV-P
- P-PDG
- PT-P
- P-TPT
- QBS
- QGL
- T-JUS-CKP
- T-MON-CKP
- T-PDG
- T-TPT

Label columns: ['class', 'state']
Timestamp stored as index: timestamp


### Checking Sensor Availability Across Recordings

Sensor availability is evaluated across all recordings before selecting a missing-value strategy.

A sensor can be:

- available throughout a recording,
- partially missing within a recording, or
- completely absent from a recording.

These cases require different handling. In particular, a sensor that is completely absent from a recording cannot be recovered through ordinary value imputation.

Each recording is therefore loaded only once, and the availability of all 27 sensor variables is checked during that single pass.

In [7]:
# Count the number of recordings containing valid data for each sensor

sensor_availability = {
    sensor: 0
    for sensor in sensor_columns
}

for file_path in parquet_files:
    # Load each recording only once
    recording = pd.read_parquet(file_path, columns=sensor_columns)

    # Check all sensors during the same file read
    for sensor in sensor_columns:
        if recording[sensor].notna().any():
            sensor_availability[sensor] += 1

# Convert the results into a Series
sensor_availability = pd.Series(
    sensor_availability,
    name="recordings_with_data"
)

# Display availability from lowest to highest
print(sensor_availability.sort_values())

P-JUS-BS            0
QBS                 0
PT-P                0
P-MON-SDV-P         0
P-MON-CKGL          2
ABER-CKGL         286
ABER-CKP          299
ESTADO-DHSV       470
ESTADO-SDV-GL     480
ESTADO-M1         501
ESTADO-M2         504
ESTADO-PXO        532
ESTADO-XO         533
ESTADO-W2         533
ESTADO-W1         558
T-PDG             563
QGL               595
ESTADO-SDV-P      661
P-JUS-CKP         665
T-MON-CKP         669
P-JUS-CKGL        839
P-ANULAR          921
T-JUS-CKP        1671
P-MON-CKP        1831
T-TPT            1929
P-PDG            1936
P-TPT            2069
Name: recordings_with_data, dtype: int64


### Understanding Sensor Availability

The sensor availability results reveal two different types of missing-data problems.

#### 1. Sensors Completely Unavailable Across the Dataset

Some sensors contain no valid values in any of the 2,228 recordings.

For example:

- `QBS` → 0 / 2,228 recordings contain valid data
- `P-JUS-BS` → 0 / 2,228
- `PT-P` → 0 / 2,228
- `P-MON-SDV-P` → 0 / 2,228

A value of `0` means that the sensor does not contain even a single valid observation anywhere in the dataset. Such variables provide no information for model training and can be excluded from the model input.

#### 2. Sensors Available in Some Recordings but Missing in Others

Other sensors contain valid data in many recordings but are completely unavailable in some recordings.

For example:

- `P-TPT` → 2,069 / 2,228 recordings contain at least one valid value
- `P-PDG` → 1,936 / 2,228
- `P-MON-CKP` → 1,831 / 2,228

For `P-TPT`, 2,069 recordings contain at least one valid value, while 159 recordings contain no valid value for that sensor.

However, availability at the recording level does not indicate that every observation within those 2,069 recordings is valid. A sensor may still contain individual missing observations inside a recording.

Therefore, two separate questions must be considered:

1. **Is the sensor available in the recording at all?**
2. **If it is available, how many individual observations are missing?**

The first question has now been evaluated across all 2,228 recordings. The next step is to investigate missing observations within recordings before deciding on an appropriate missing-value treatment.

#### Key Distinction

```text
QBS
↓
0 / 2,228 recordings with data
↓
Completely unavailable
↓
No meaningful imputation is possible
↓
Candidate for removal


P-TPT
↓
2,069 / 2,228 recordings with data
↓
Available in most recordings
↓
159 recordings have no data
↓
Individual missing values may still exist
↓
Requires further missingness analysis

### Identifying Completely Unavailable Sensors

The previous analysis identified sensors that are completely unavailable in some or all recordings.

A sensor with no valid observations across the entire dataset provides no usable information and can be removed from the model input. Sensors that are available in at least some recordings are retained for further analysis.

The following sensors have no valid observations anywhere in the dataset:

- `QBS`
- `P-JUS-BS`
- `PT-P`
- `P-MON-SDV-P`

These variables will be excluded from the sensor feature set. The remaining sensors will be investigated for partial missingness before an imputation strategy is selected.

In [8]:
# Identify sensors that contain no valid observations anywhere in the dataset

unavailable_sensors = sensor_availability[
    sensor_availability == 0
].index.tolist()

# Keep sensors that contain at least one valid observation
available_sensors = [
    sensor
    for sensor in sensor_columns
    if sensor not in unavailable_sensors
]

print("Completely unavailable sensors:")
for sensor in unavailable_sensors:
    print("-", sensor)

print("\nNumber of completely unavailable sensors:", len(unavailable_sensors))
print("Number of remaining sensors:", len(available_sensors))

Completely unavailable sensors:
- P-JUS-BS
- P-MON-SDV-P
- PT-P
- QBS

Number of completely unavailable sensors: 4
Number of remaining sensors: 23


### Measuring Partial Missingness

The remaining 23 sensors contain at least some valid observations in the dataset, but this does not guarantee complete coverage.

For each remaining sensor, the proportion of missing observations is calculated across all recordings. This provides a dataset-level view of how frequently each sensor is unavailable at the individual observation level.

Sensors with substantial missingness will be examined further rather than being removed or imputed solely on the basis of this overall percentage.

In [9]:
# Calculate the total number of observations and missing values for each
# remaining sensor across the complete dataset

sensor_missing_values = {
    sensor: 0
    for sensor in available_sensors
}

sensor_total_values = {
    sensor: 0
    for sensor in available_sensors
}

for file_path in parquet_files:
    # Load only the remaining sensor variables
    recording = pd.read_parquet(
        file_path,
        columns=available_sensors
    )

    # Count total and missing observations for each sensor
    for sensor in available_sensors:
        sensor_total_values[sensor] += len(recording)
        sensor_missing_values[sensor] += recording[sensor].isna().sum()

# Create a summary table
missingness_summary = pd.DataFrame(
    {
        "total_observations": sensor_total_values,
        "missing_observations": sensor_missing_values
    }
)

# Calculate the percentage of missing observations
missingness_summary["missing_percentage"] = (
    missingness_summary["missing_observations"]
    / missingness_summary["total_observations"]
    * 100
)

# Display sensors from highest to lowest missingness
missingness_summary = missingness_summary.sort_values(
    "missing_percentage",
    ascending=False
)

missingness_summary

,total_observations,missing_observations,missing_percentage
P-MON-CKGL,76587318,76278592,99.596897
ABER-CKGL,76587318,68095805,88.912638
ABER-CKP,76587318,64260860,83.905354
ESTADO-DHSV,76587318,62721921,81.895962
ESTADO-SDV-GL,76587318,59482047,77.665661
ESTADO-M2,76587318,58899036,76.904424
ESTADO-M1,76587318,58814113,76.793540
ESTADO-PXO,76587318,58411115,76.267346
ESTADO-W2,76587318,58293856,76.114241
ESTADO-XO,76587318,58048588,75.793995


### Comparing Missingness Across Event Categories

Dataset-level missingness can conceal important differences between event categories. A sensor with high overall missingness may still be well represented within particular event categories.

A folder-by-sensor missingness table is therefore created with:

- **Rows:** the 10 event folders (`0`–`9`)
- **Columns:** the 23 remaining sensors
- **Values:** percentage of missing observations for each sensor within each folder

This provides a direct comparison of sensor availability across event categories.

In [10]:
folder_missingness = []

for folder in range(10):
    folder_files = recording_inventory[
        recording_inventory["folder"] == folder
    ]["file_path"]

    for file_path in folder_files:
        recording = pd.read_parquet(
            file_path,
            columns=available_sensors
        )

        for sensor in available_sensors:
            missing_percentage = (
                recording[sensor].isna().mean() * 100
            )

            folder_missingness.append(
                {
                    "folder": folder,
                    "sensor": sensor,
                    "missing_percentage": missing_percentage
                }
            )

folder_missingness = pd.DataFrame(folder_missingness)

folder_missingness.head()

,folder,sensor,missing_percentage
0,0,ABER-CKGL,100.0
1,0,ABER-CKP,100.0
2,0,ESTADO-DHSV,0.0
3,0,ESTADO-M1,0.0
4,0,ESTADO-M2,0.0


### Screening Sensors by Overall Data Availability

The overall availability of the remaining sensors is calculated as the proportion of non-missing observations across the complete dataset.

This provides an initial screening measure for identifying sensors with very limited usable data.

The availability percentage is used only as a diagnostic at this stage. A sensor will not be removed solely because of a high missing percentage, since the folder-level analysis has shown that sensor availability can vary substantially between event categories.

In [11]:
# Calculate the percentage of non-missing observations for each sensor

availability_summary = missingness_summary.copy()

availability_summary["availability_percentage"] = (
    100 - availability_summary["missing_percentage"]
)

# Display sensors from highest to lowest availability
availability_summary = availability_summary.sort_values(
    "availability_percentage",
    ascending=False
)

availability_summary[
    ["total_observations", "missing_observations", "availability_percentage"]
]

,total_observations,missing_observations,availability_percentage
P-TPT,76587318,5381783,92.973010
P-MON-CKP,76587318,7079851,90.755844
P-PDG,76587318,7916216,89.663803
T-TPT,76587318,10262040,86.600863
T-JUS-CKP,76587318,16032900,79.065855
P-JUS-CKGL,76587318,44691084,41.646887
P-ANULAR,76587318,51126803,33.243774
QGL,76587318,52021358,32.075754
ESTADO-SDV-P,76587318,52271211,31.749522
T-MON-CKP,76587318,52969713,30.837488


### Examining the Temporal Pattern of Missing Values

Overall missingness percentages do not indicate how missing observations are distributed over time.

For time-series data, missing values may occur as isolated gaps or as continuous blocks. The distinction is important because long missing intervals can represent periods in which a sensor was unavailable, rather than individual measurement errors.

A representative recording is therefore examined at the observation level to characterize the temporal pattern of missing values.

In [12]:
# Select a representative recording with commonly available sensors

sample_file = parquet_files[0]

sample_recording = pd.read_parquet(
    sample_file,
    columns=available_sensors
)

# Calculate the number of missing values for each sensor
sample_missingness = sample_recording.isna().sum()

# Display sensors with missing values in this recording
sample_missingness = sample_missingness[
    sample_missingness > 0
].sort_values(ascending=False)

print("Recording:", sample_file.name)
print("\nMissing observations by sensor:")

sample_missingness

Recording: WELL-00001_20170201010207.parquet

Missing observations by sensor:


ABER-CKGL     21474
ABER-CKP      21474
P-JUS-CKP     21474
P-MON-CKGL    21474
T-MON-CKP     21474
dtype: int64

### Interpreting Missing Sensors Within a Recording

The selected recording contains 23 sensor variables. The output displays only the sensors that have at least one missing observation.

For this recording, only 5 sensors contain missing values:

- `ABER-CKGL`
- `ABER-CKP`
- `P-JUS-CKP`
- `P-MON-CKGL`
- `T-MON-CKP`

Each of these sensors has 21,474 missing observations, which is equal to the total number of observations in the recording. Therefore, these five sensors are **100% missing within this particular recording**.

The remaining 18 sensors are not displayed because they contain zero missing observations in this recording.

This highlights an important characteristic of the dataset: a sensor can have usable observations across the overall dataset while being completely unavailable within an individual recording.

Therefore, missingness needs to be considered at both the **dataset level** and the **recording level** before selecting a missing-value treatment.

### Identifying Continuous Missing Blocks

A sensor can be missing for isolated observations or for continuous periods of time.

Continuous missing blocks are particularly important in time-series preprocessing because they may indicate that a sensor was unavailable for a sustained period rather than experiencing occasional measurement gaps.

The length of consecutive missing periods is therefore examined for the selected recording. This helps distinguish short gaps that may be suitable for interpolation from long periods where the sensor should be treated as unavailable.

In [13]:
# Calculate the lengths of consecutive missing blocks for each sensor

missing_blocks = {}

for sensor in available_sensors:
    # Identify missing observations
    is_missing = sample_recording[sensor].isna()

    # Create groups whenever the missing/non-missing status changes
    groups = is_missing.ne(is_missing.shift()).cumsum()

    # Count the length of each consecutive missing group
    block_lengths = is_missing.groupby(groups).sum()

    # Keep only groups that actually contain missing values
    block_lengths = block_lengths[block_lengths > 0]

    missing_blocks[sensor] = block_lengths.tolist()

# Display the longest missing block for each sensor
longest_missing_blocks = {
    sensor: max(block_lengths, default=0)
    for sensor, block_lengths in missing_blocks.items()
}

longest_missing_blocks = pd.Series(
    longest_missing_blocks,
    name="longest_missing_block"
).sort_values(ascending=False)

longest_missing_blocks 

ABER-CKGL        21474
ABER-CKP         21474
P-JUS-CKP        21474
P-MON-CKGL       21474
T-MON-CKP        21474
ESTADO-M1            0
ESTADO-DHSV          0
ESTADO-SDV-GL        0
ESTADO-PXO           0
ESTADO-M2            0
ESTADO-SDV-P         0
ESTADO-XO            0
ESTADO-W2            0
P-JUS-CKGL           0
ESTADO-W1            0
P-ANULAR             0
P-MON-CKP            0
P-TPT                0
P-PDG                0
QGL                  0
T-JUS-CKP            0
T-PDG                0
T-TPT                0
Name: longest_missing_block, dtype: int64

### Interpreting Continuous Missing Blocks

The longest missing block represents the maximum number of consecutive observations for which a sensor has missing values within the selected recording.

For this recording, the following five sensors each have a longest missing block of 21,474 observations:

- `ABER-CKGL`
- `ABER-CKP`
- `P-JUS-CKP`
- `P-MON-CKGL`
- `T-MON-CKP`

Since the recording contains exactly 21,474 observations, these sensors are missing for the **entire duration of the recording**.

The remaining 18 sensors have a longest missing block of `0`, meaning they contain no missing observations anywhere in this recording.

This demonstrates that missingness in the 3W dataset can occur as a **complete absence of a sensor for an entire recording**, rather than only as isolated missing observations.

For a sensor that is completely absent from a recording, interpolation is not appropriate because there are no valid observations within that recording from which to estimate the missing values.

The next analysis will therefore examine a recording where a sensor contains both valid and missing observations, allowing intermittent missing gaps to be distinguished from complete sensor absence.

### Distinguishing Complete Absence from Intermittent Missingness

The continuous-block analysis shows that missing sensor data can occur in two fundamentally different forms.

For the selected recording, five sensors have a continuous missing block equal to the full recording length. These sensors are therefore completely unavailable throughout the recording.

The remaining sensors have no missing blocks in this recording.

This confirms that a missing sensor does not necessarily represent an isolated measurement gap. Some sensors are absent for an entire recording, making interpolation inappropriate because no valid observations are available within that recording.

Further analysis will examine recordings where sensors contain both valid and missing observations to determine how intermittent gaps should be handled.

In [14]:
# Find a recording that contains both valid and missing observations
# for at least one sensor

mixed_missing_file = None

for file_path in parquet_files:
    recording = pd.read_parquet(
        file_path,
        columns=available_sensors
    )

    # Check whether any sensor has both valid and missing observations
    has_mixed_missingness = False

    for sensor in available_sensors:
        has_missing = recording[sensor].isna().any()
        has_valid = recording[sensor].notna().any()

        if has_missing and has_valid:
            has_mixed_missingness = True
            break

    if has_mixed_missingness:
        mixed_missing_file = file_path
        break

print("Selected recording:")
print(mixed_missing_file.name)

Selected recording:
WELL-00002_20170614120000.parquet


### Examining Intermittent Missingness

The selected recording contains at least one sensor with both valid and missing observations.

The missing-value pattern is examined for this recording to determine which sensors contain intermittent gaps and how frequently those gaps occur.

This distinction is important because short gaps surrounded by valid observations may be handled differently from long periods of consecutive missing values.

In [15]:
# Load the selected recording

mixed_missing_recording = pd.read_parquet(
    mixed_missing_file,
    columns=available_sensors
)

# Calculate missing and valid observations for each sensor
mixed_missing_summary = pd.DataFrame(
    {
        "missing_observations": mixed_missing_recording.isna().sum(),
        "valid_observations": mixed_missing_recording.notna().sum()
    }
)

# Keep only sensors that contain both missing and valid observations
mixed_missing_summary = mixed_missing_summary[
    (mixed_missing_summary["missing_observations"] > 0)
    & (mixed_missing_summary["valid_observations"] > 0)
]

# Calculate the missing percentage
mixed_missing_summary["missing_percentage"] = (
    mixed_missing_summary["missing_observations"]
    / len(mixed_missing_recording)
    * 100
)

# Display the results
mixed_missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

,missing_observations,valid_observations,missing_percentage
P-ANULAR,299,21184,1.391798
P-PDG,299,21184,1.391798
P-TPT,299,21184,1.391798
T-TPT,299,21184,1.391798


### Interpreting Intermittent Missingness

The selected recording contains four sensors with both valid and missing observations:

- `P-ANULAR`
- `P-PDG`
- `P-TPT`
- `T-TPT`

Each sensor contains 21,184 valid observations and 299 missing observations, corresponding to 1.39% missingness within this recording.

Unlike a completely unavailable sensor, these variables contain sufficient valid observations within the same recording to examine the temporal structure of their missing values.

The next step is to determine whether the 299 missing observations occur as isolated gaps or as continuous missing blocks. This distinction is necessary before selecting an appropriate treatment for intermittent missing values.

In [16]:
# Calculate the consecutive missing blocks for the sensors
# with intermittent missingness

intermittent_sensors = mixed_missing_summary.index.tolist()

intermittent_missing_blocks = {}

for sensor in intermittent_sensors:
    # Identify missing observations
    is_missing = mixed_missing_recording[sensor].isna()

    # Create a new group whenever the missing/non-missing status changes
    groups = is_missing.ne(is_missing.shift()).cumsum()

    # Calculate the length of each consecutive missing block
    block_lengths = is_missing.groupby(groups).sum()

    # Keep only groups containing missing observations
    block_lengths = block_lengths[block_lengths > 0]

    intermittent_missing_blocks[sensor] = block_lengths.tolist()

# Display all missing block lengths
intermittent_missing_blocks

{'P-ANULAR': [299], 'P-PDG': [299], 'P-TPT': [299], 'T-TPT': [299]}

### Interpreting Intermittent Missingness

The selected recording contains four sensors with both valid and missing observations:

- `P-ANULAR`
- `P-PDG`
- `P-TPT`
- `T-TPT`

Each sensor contains 21,184 valid observations and 299 missing observations, corresponding to approximately 1.39% missingness within this recording.

The missing-value block analysis shows that all 299 missing observations occur as **one continuous block** for each of these four sensors.

Therefore:

- `P-ANULAR` → one continuous missing block of 299 observations
- `P-PDG` → one continuous missing block of 299 observations
- `P-TPT` → one continuous missing block of 299 observations
- `T-TPT` → one continuous missing block of 299 observations

This means the missing observations are not scattered randomly throughout the recording. Instead, each of these sensors becomes unavailable for one continuous period.

This is different from isolated missing observations and is important when selecting an appropriate preprocessing strategy. The next step is to determine where this continuous missing block occurs in the recording timeline, such as at the beginning, middle, or end of the recording.

### Locating the Continuous Missing Block

The previous analysis established that the 299 missing observations form one continuous block for each of the four affected sensors.

The position of this block within the recording is now identified using the timestamps of the first and last missing observations.

This determines whether the missing period occurs at the beginning, middle, or end of the recording. The location of the gap is important because a missing block surrounded by valid observations can potentially be treated differently from a missing period at the boundary of a recording.

In [17]:
missing_block_locations = {}

for sensor in intermittent_sensors:
    missing_timestamps = mixed_missing_recording.index[
        mixed_missing_recording[sensor].isna()
    ]

    missing_block_locations[sensor] = {
        "start": missing_timestamps.min(),
        "end": missing_timestamps.max(),
        "observations": len(missing_timestamps)
    }

missing_block_locations

{'P-ANULAR': {'start': Timestamp('2017-06-14 14:09:39'),
  'end': Timestamp('2017-06-14 14:14:37'),
  'observations': 299},
 'P-PDG': {'start': Timestamp('2017-06-14 14:09:39'),
  'end': Timestamp('2017-06-14 14:14:37'),
  'observations': 299},
 'P-TPT': {'start': Timestamp('2017-06-14 14:09:39'),
  'end': Timestamp('2017-06-14 14:14:37'),
  'observations': 299},
 'T-TPT': {'start': Timestamp('2017-06-14 14:09:39'),
  'end': Timestamp('2017-06-14 14:14:37'),
  'observations': 299}}

In [18]:
missing_block_locations_df = pd.DataFrame.from_dict(
    missing_block_locations,
    orient="index"
)

missing_block_locations_df.index.name = "sensor"
missing_block_locations_df = missing_block_locations_df.reset_index()

missing_block_locations_df

,sensor,start,end,observations
0,P-ANULAR,2017-06-14 14:09:39,2017-06-14 14:14:37,299
1,P-PDG,2017-06-14 14:09:39,2017-06-14 14:14:37,299
2,P-TPT,2017-06-14 14:09:39,2017-06-14 14:14:37,299
3,T-TPT,2017-06-14 14:09:39,2017-06-14 14:14:37,299


### Determining Whether the Missing Block Is Internal or at a Boundary

The missing block occurs from `2017-06-14 14:09:39` to `2017-06-14 14:14:37` and contains 299 consecutive observations.

The position of this block within the complete recording is determined by comparing its start and end timestamps with the recording's first and last timestamps.

If valid observations exist both before and after the missing block, the gap is an **internal missing block**. If the block begins at the first timestamp or ends at the last timestamp, it is a boundary missing block.

An internal gap is particularly important for time-series preprocessing because valid observations are available on both sides of the missing period.

In [19]:
recording_start = mixed_missing_recording.index.min()
recording_end = mixed_missing_recording.index.max()

missing_block_boundary_check = missing_block_locations_df.copy()

missing_block_boundary_check["is_at_start"] = (
    missing_block_boundary_check["start"] == recording_start
)

missing_block_boundary_check["is_at_end"] = (
    missing_block_boundary_check["end"] == recording_end
)

missing_block_boundary_check["is_internal"] = (
    ~missing_block_boundary_check["is_at_start"]
    & ~missing_block_boundary_check["is_at_end"]
)

missing_block_boundary_check[
    [
        "sensor",
        "start",
        "end",
        "observations",
        "is_at_start",
        "is_at_end",
        "is_internal"
    ]
]

,sensor,start,end,observations,is_at_start,is_at_end,is_internal
0,P-ANULAR,2017-06-14 14:09:39,2017-06-14 14:14:37,299,False,False,True
1,P-PDG,2017-06-14 14:09:39,2017-06-14 14:14:37,299,False,False,True
2,P-TPT,2017-06-14 14:09:39,2017-06-14 14:14:37,299,False,False,True
3,T-TPT,2017-06-14 14:09:39,2017-06-14 14:14:37,299,False,False,True


### Examining the Observations Surrounding the Missing Block

The missing block is internal to the recording, meaning valid observations are available immediately before and after the gap.

The observations surrounding the gap are examined to determine the structure of the transition into and out of the missing period.

This provides a closer view of the local time-series pattern and helps determine whether the missing block is isolated between valid measurements.

In [20]:
first_missing_time = missing_block_locations_df["start"].min()
last_missing_time = missing_block_locations_df["end"].max()

observations_before = mixed_missing_recording.loc[
    mixed_missing_recording.index < first_missing_time
].tail(5)

observations_after = mixed_missing_recording.loc[
    mixed_missing_recording.index > last_missing_time
].head(5)

print("Observations immediately before the missing block:")
display(observations_before[intermittent_sensors])

print("\nObservations immediately after the missing block:")
display(observations_after[intermittent_sensors])

Observations immediately before the missing block:


,P-ANULAR,P-PDG,P-TPT,T-TPT
timestamp,,,,
2017-06-14 14:09:34,12089440.0,0.0,8757864.0,118.3171
2017-06-14 14:09:35,12089440.0,0.0,8757864.0,118.3171
2017-06-14 14:09:36,12089440.0,0.0,8757864.0,118.3171
2017-06-14 14:09:37,12089440.0,0.0,8757864.0,118.3171
2017-06-14 14:09:38,12089440.0,0.0,8757864.0,118.3171



Observations immediately after the missing block:


,P-ANULAR,P-PDG,P-TPT,T-TPT
timestamp,,,,
2017-06-14 14:14:38,12109390.0,0.0,8777813.0,118.3171
2017-06-14 14:14:39,12109520.0,0.0,8777759.0,118.3170
2017-06-14 14:14:40,12109640.0,0.0,8777705.0,118.3169
2017-06-14 14:14:41,12109770.0,0.0,8777651.0,118.3167
2017-06-14 14:14:42,12109890.0,0.0,8777597.0,118.3166


### Checking Whether Multiple Sensors Share the Same Missing Period

The four sensors identified in the selected recording become missing at the same timestamp and resume at the same timestamp.

This is examined across the recording to determine whether their missing observations always occur together.

If multiple sensors share the same missing interval, the missingness may represent a common data-availability issue rather than independent sensor failures. This distinction is important when designing the preprocessing strategy.

In [21]:
missing_pattern = mixed_missing_recording[intermittent_sensors].isna()

same_missing_pattern = (
    missing_pattern.eq(missing_pattern.iloc[:, 0], axis=0).all(axis=1)
)

print("All four sensors have the same missing/valid status:", same_missing_pattern.all())

print("\nNumber of observations where all four sensors are missing:",
      missing_pattern.all(axis=1).sum())

print("Number of observations where at least one sensor is missing:",
      missing_pattern.any(axis=1).sum())

All four sensors have the same missing/valid status: True

Number of observations where all four sensors are missing: 299
Number of observations where at least one sensor is missing: 299


### Checking the Sampling Interval Around the Missing Block

The missing block contains 299 observations, and the timestamps before and after the gap suggest one-second sampling.

The timestamp difference between the last valid observation before the gap and the first valid observation after the gap is examined to determine the actual duration represented by the missing period.

This verifies whether the 299 missing observations correspond to a continuous 299-second interval and confirms the temporal structure of the gap.

In [22]:
first_missing_time = missing_block_locations_df["start"].min()
last_missing_time = missing_block_locations_df["end"].max()

last_valid_before_gap = mixed_missing_recording.index[
    (
        mixed_missing_recording.index < first_missing_time
    )
    & (
        mixed_missing_recording[intermittent_sensors[0]].notna()
    )
].max()

first_valid_after_gap = mixed_missing_recording.index[
    (
        mixed_missing_recording.index > last_missing_time
    )
    & (
        mixed_missing_recording[intermittent_sensors[0]].notna()
    )
].min()

time_difference = (
    first_valid_after_gap - last_valid_before_gap
)

print("Last valid timestamp before gap:", last_valid_before_gap)
print("First valid timestamp after gap:", first_valid_after_gap)
print("Time difference:", time_difference)

Last valid timestamp before gap: 2017-06-14 14:09:38
First valid timestamp after gap: 2017-06-14 14:14:38
Time difference: 0 days 00:05:00


### Summarizing the Missing-Data Pattern

The analysis of the selected recording shows that missing values can occur as a coordinated, internal gap affecting multiple sensors simultaneously.

For this recording, four sensors are missing for the same 299-observation interval. The gap is internal to the recording and spans approximately five minutes between the surrounding valid timestamps.

This indicates that the missing values are structured rather than isolated random observations.

Before selecting an imputation strategy, the same type of missingness should be examined across a broader set of recordings to determine whether coordinated missing blocks are common throughout the dataset.

In [23]:
print("Recording analyzed:", mixed_missing_file.name)
print("Affected sensors:", len(intermittent_sensors))
print("Missing observations per sensor:", 299)
print("Missing block duration:", time_difference)
print("Gap is internal:", True)
print("Sensors missing simultaneously:", intermittent_sensors)

Recording analyzed: WELL-00002_20170614120000.parquet
Affected sensors: 4
Missing observations per sensor: 299
Missing block duration: 0 days 00:05:00
Gap is internal: True
Sensors missing simultaneously: ['P-ANULAR', 'P-PDG', 'P-TPT', 'T-TPT']


### Checking How Common Coordinated Missingness Is

The selected recording demonstrated a five-minute internal gap shared by four sensors.

To determine whether this pattern is specific to one recording or occurs more broadly, a sample of recordings is examined for observations where multiple sensors are missing at the same timestamp.

The number of simultaneously missing sensors is counted for each observation. This provides an initial indication of how frequently coordinated missingness occurs across recordings.

In [24]:
sample_files_for_missingness = parquet_files[:20]

coordinated_missingness = []

for file_path in sample_files_for_missingness:
    recording = pd.read_parquet(
        file_path,
        columns=available_sensors
    )

    missing_sensor_count = recording.isna().sum(axis=1)

    coordinated_missingness.append(
        {
            "recording_id": file_path.stem,
            "observations_with_missing_data": (missing_sensor_count > 0).sum(),
            "observations_with_multiple_missing_sensors": (
                missing_sensor_count > 1
            ).sum(),
            "maximum_simultaneously_missing_sensors": missing_sensor_count.max()
        }
    )

coordinated_missingness_df = pd.DataFrame(coordinated_missingness)

coordinated_missingness_df

,recording_id,observations_with_missing_data,observations_with_multiple_missing_sensors,maximum_simultaneously_missing_sensors
0,WELL-00001_20170201010207,21474,21474,5
1,WELL-00001_20170201060114,21527,21527,5
2,WELL-00001_20170201110124,21517,21517,5
3,WELL-00001_20170201160311,21410,21410,5
4,WELL-00001_20170201210228,21453,21453,5
5,WELL-00001_20170202020343,21378,21378,5
6,WELL-00001_20170202070239,21442,21442,5
7,WELL-00001_20170218000146,21495,21495,5
8,WELL-00001_20170218050218,21463,21463,5
9,WELL-00001_20170218100218,21463,21463,5


### Interpreting Coordinated Missingness Across Recordings

The analysis was performed on the first 20 recordings to determine whether multiple sensors are missing at the same timestamps.

For every recording in this sample:

- `observations_with_missing_data` and `observations_with_multiple_missing_sensors` have exactly the same value.
- `maximum_simultaneously_missing_sensors` is `5`.

This means that whenever a missing observation occurs in these recordings, **multiple sensors are missing at the same timestamp**, rather than only one sensor being unavailable.

For example, in `WELL-00001_20170201010207`:

- 21,474 observations contain missing sensor values.
- All 21,474 of those observations have multiple sensors missing simultaneously.
- At some timestamps, as many as 5 sensors are missing at the same time.

The same pattern appears consistently across all 20 sampled recordings.

This provides evidence that missingness in the 3W dataset is often **structured and coordinated across sensors**, rather than consisting primarily of isolated missing values.

However, this analysis covers only the first 20 recordings. A complete dataset-level conclusion should therefore be based on the full set of 2,228 recordings before the final preprocessing strategy is selected.

### Quantifying Coordinated Missingness Across the Complete Dataset

The sample analysis indicates that missing observations are frequently shared across multiple sensors.

To determine whether this pattern is representative of the complete dataset, the same analysis is now performed across all 2,228 recordings.

For each recording, the number of observations with multiple sensors missing simultaneously is compared with the total number of observations containing any missing sensor value.

This provides a dataset-wide measure of how often missingness occurs as a coordinated event rather than as an isolated sensor-level gap.

In [25]:
coordinated_missingness_summary = []

for file_path in parquet_files:
    recording = pd.read_parquet(
        file_path,
        columns=available_sensors
    )

    missing_sensor_count = recording.isna().sum(axis=1)

    observations_with_missing = (
        missing_sensor_count > 0
    ).sum()

    observations_with_multiple_missing = (
        missing_sensor_count > 1
    ).sum()

    coordinated_missingness_summary.append(
        {
            "recording_id": file_path.stem,
            "folder": int(file_path.parent.name),
            "observations_with_missing_data": observations_with_missing,
            "observations_with_multiple_missing_sensors": (
                observations_with_multiple_missing
            ),
            "maximum_simultaneously_missing_sensors": (
                missing_sensor_count.max()
            )
        }
    )

coordinated_missingness_df = pd.DataFrame(
    coordinated_missingness_summary
)

coordinated_missingness_df.head(20)

,recording_id,folder,observations_with_missing_data,observations_with_multiple_missing_sensors,maximum_simultaneously_missing_sensors
0,WELL-00001_20170201010207,0,21474,21474,5
1,WELL-00001_20170201060114,0,21527,21527,5
2,WELL-00001_20170201110124,0,21517,21517,5
3,WELL-00001_20170201160311,0,21410,21410,5
4,WELL-00001_20170201210228,0,21453,21453,5
5,WELL-00001_20170202020343,0,21378,21378,5
6,WELL-00001_20170202070239,0,21442,21442,5
7,WELL-00001_20170218000146,0,21495,21495,5
8,WELL-00001_20170218050218,0,21463,21463,5
9,WELL-00001_20170218100218,0,21463,21463,5


### Interpreting the Recording-Level Coordinated Missingness

The analysis examined the first 20 recordings in the dataset and summarized the missing-data pattern for each recording.

All 20 recordings contain observations with missing sensor values. In every recording, the number of observations with missing data is exactly equal to the number of observations with multiple missing sensors.

This means that, within these recordings, missing observations consistently involve **multiple sensors becoming unavailable at the same timestamp**, rather than individual sensors having isolated missing values.

The `maximum_simultaneously_missing_sensors` value is `5` for every recording. Therefore, at some point in each of these recordings, five sensors are missing simultaneously.

For example, the first recording contains 21,474 observations with missing sensor data, and all 21,474 of those observations have multiple sensors missing at the same time.

However, all 20 displayed recordings belong to **Folder 0**. Therefore, this output demonstrates coordinated missingness within the displayed recordings but cannot yet be used to determine whether the same pattern occurs across the other event categories.

The next analysis summarizes these results across all folders to examine whether coordinated missingness varies between event categories.

### Comparing Coordinated Missingness Across Event Categories

The complete recording-level analysis contains information for all 2,228 recordings.

Because the recordings belong to different event categories, coordinated missingness is summarized by folder to determine whether the pattern varies between event categories.

For each folder, the number of recordings containing missing observations, the number containing multiple simultaneously missing sensors, and the maximum number of simultaneously missing sensors are summarized.

This provides a folder-level view of the missing-data structure across the complete dataset.

In [26]:
folder_coordinated_missingness = (
    coordinated_missingness_df
    .groupby("folder")
    .agg(
        recordings_with_missing_data=(
            "observations_with_missing_data",
            lambda values: (values > 0).sum()
        ),
        recordings_with_multiple_missing_sensors=(
            "observations_with_multiple_missing_sensors",
            lambda values: (values > 0).sum()
        ),
        maximum_simultaneously_missing_sensors=(
            "maximum_simultaneously_missing_sensors",
            "max"
        )
    )
    .reset_index()
)

folder_coordinated_missingness

,folder,recordings_with_missing_data,recordings_with_multiple_missing_sensors,maximum_simultaneously_missing_sensors
0,0,594,594,23
1,1,128,128,19
2,2,38,38,22
3,3,106,106,23
4,4,343,343,23
5,5,450,450,23
6,6,221,221,19
7,7,46,46,23
8,8,95,95,23
9,9,207,207,23


### Interpreting Coordinated Missingness Across Event Categories

The analysis summarizes coordinated missingness across all 10 folders in the dataset.

For every folder, the number of recordings with missing sensor observations is exactly equal to the number of recordings with multiple missing sensors. This indicates that, whenever missing sensor data occurs within these recordings, it involves multiple sensors simultaneously rather than only a single sensor.

The number of affected recordings varies substantially between folders:

- **Folder 0:** 594 recordings contain missing data, with up to 23 sensors missing simultaneously.
- **Folder 1:** 128 recordings contain missing data, with up to 19 sensors missing simultaneously.
- **Folder 2:** 38 recordings contain missing data, with up to 22 sensors missing simultaneously.
- **Folder 3:** 106 recordings contain missing data, with up to 23 sensors missing simultaneously.
- **Folder 4:** 343 recordings contain missing data, with up to 23 sensors missing simultaneously.
- **Folder 5:** 450 recordings contain missing data, with up to 23 sensors missing simultaneously.
- **Folder 6:** 221 recordings contain missing data, with up to 19 sensors missing simultaneously.
- **Folder 7:** 46 recordings contain missing data, with up to 23 sensors missing simultaneously.
- **Folder 8:** 95 recordings contain missing data, with up to 23 sensors missing simultaneously.
- **Folder 9:** 207 recordings contain missing data, with up to 23 sensors missing simultaneously.

The maximum value of 23 indicates that, in some recordings within those folders, **all 23 retained sensors are missing at the same timestamp**.

This confirms that coordinated missingness is not limited to a single event category. It occurs across all 10 folders, although the number of affected recordings and the maximum number of simultaneously missing sensors differ between folders.

The results reinforce that missing values in the 3W dataset should be treated as a **structured time-series data-availability issue**, rather than assuming that every missing value represents an independent measurement error.

The next step is to examine the **duration of these coordinated missing periods** across recordings, since the length of a missing block will be important when determining how it should be handled during preprocessing.

### Analyzing Class and State Labels

The `class` and `state` columns represent label-related information associated with each time-series observation.

Before preprocessing these columns, their values must be examined across all files and instances rather than relying on a single example.

The analysis will determine:

- Which `class` values occur throughout the dataset.
- How frequently each class value occurs.
- Which `state` values occur throughout the dataset.
- Whether missing labels are present.
- Whether special or less common label values occur only in particular event categories.

This information is required before defining the target variable and deciding how different label values should be handled during feature engineering and model development.

In [27]:
class_counts = {}
state_counts = {}

for file_path in parquet_files:
    labels = pd.read_parquet(
        file_path,
        columns=["class", "state"]
    )

    for class_value, count in labels["class"].value_counts(dropna=False).items():
        class_counts[class_value] = (
            class_counts.get(class_value, 0) + count
        )

    for state_value, count in labels["state"].value_counts(dropna=False).items():
        state_counts[state_value] = (
            state_counts.get(state_value, 0) + count
        )

class_counts_df = pd.DataFrame(
    {
        "class": list(class_counts.keys()),
        "observations": list(class_counts.values())
    }
).sort_values("class")

state_counts_df = pd.DataFrame(
    {
        "state": list(state_counts.keys()),
        "observations": list(state_counts.values())
    }
).sort_values("state")

print("Class values across the complete dataset:")
display(class_counts_df)

print("\nState values across the complete dataset:")
display(state_counts_df)

Class values across the complete dataset:


,class,observations
0,0,17305269
3,1,2909887
4,2,366858
6,3,4834079
7,4,2454883
8,5,10553279
10,6,3879083
13,7,138148
14,8,744061
16,9,3203605



State values across the complete dataset:


,state,observations
0,0,69660362
3,1,1867665
9,2,83107
5,3,44752
4,4,40095
7,5,22214
8,6,566442
2,7,225990
6,8,48291
1,<NA>,4028400


### Measuring Missing Class and State Labels

The previous analysis identified the distinct `class` and `state` values across all files and instances.

The presence of missing label values must also be quantified because missing target information cannot be used directly for supervised machine learning.

The number and percentage of missing values in `class` and `state` are therefore calculated across the complete dataset. This provides a dataset-level view of label completeness before deciding how these observations should be handled.

In [28]:
total_observations = 0
missing_class = 0
missing_state = 0

for file_path in parquet_files:
    labels = pd.read_parquet(
        file_path,
        columns=["class", "state"]
    )

    total_observations += len(labels)

    missing_class += labels["class"].isna().sum()
    missing_state += labels["state"].isna().sum()

label_missingness = pd.DataFrame(
    {
        "label": ["class", "state"],
        "missing_observations": [
            missing_class,
            missing_state
        ]
    }
)

label_missingness["missing_percentage"] = (
    label_missingness["missing_observations"]
    / total_observations
    * 100
)

label_missingness

,label,missing_observations,missing_percentage
0,class,4028400,5.259879
1,state,4028400,5.259879


### Examining Missing Labels by Event Category

The previous analysis showed that approximately 5.26% of observations have missing values in both `class` and `state`.

The distribution of these missing labels is now summarized across the 10 event-category folders.

For each folder, the total number of observations and the number of missing `class` and `state` labels are calculated. The corresponding missing percentages are then used to compare label completeness between event categories.

This analysis helps determine whether missing labels are distributed broadly across the dataset or are concentrated within particular event categories.

In [29]:
folder_label_missingness = []

for folder in range(10):
    folder_files = recording_inventory[
        recording_inventory["folder"] == folder
    ]["file_path"]

    total_observations = 0
    missing_class = 0
    missing_state = 0

    for file_path in folder_files:
        labels = pd.read_parquet(
            file_path,
            columns=["class", "state"]
        )

        total_observations += len(labels)
        missing_class += labels["class"].isna().sum()
        missing_state += labels["state"].isna().sum()

    folder_label_missingness.append(
        {
            "folder": folder,
            "total_observations": total_observations,
            "missing_class": missing_class,
            "missing_state": missing_state
        }
    )

folder_label_missingness_summary = pd.DataFrame(
    folder_label_missingness
)

folder_label_missingness_summary["class_missing_percentage"] = (
    folder_label_missingness_summary["missing_class"]
    / folder_label_missingness_summary["total_observations"]
    * 100
)

folder_label_missingness_summary["state_missing_percentage"] = (
    folder_label_missingness_summary["missing_state"]
    / folder_label_missingness_summary["total_observations"]
    * 100
)

folder_label_missingness_summary

,folder,total_observations,missing_class,missing_state,class_missing_percentage,state_missing_percentage
0,0,12158183,2138400,2138400,17.588154,17.588154
1,1,9107107,14400,14400,0.158118,0.158118
2,2,737785,79200,79200,10.734835,10.734835
3,3,4949279,115200,115200,2.327612,2.327612
4,4,3689683,1234800,1234800,33.466290,33.466290
5,5,13301677,39600,39600,0.297707,0.297707
6,6,5882267,21600,21600,0.367205,0.367205
7,7,10284155,129600,129600,1.260191,1.260191
8,8,6995955,50400,50400,0.720416,0.720416
9,9,9481227,205200,205200,2.164277,2.164277


### Interpreting Missing Class and State Labels Across Event Categories

The analysis summarizes the completeness of the `class` and `state` labels across all 10 event-category folders.

The missing percentages for `class` and `state` are identical in every folder. This is consistent with the earlier dataset-level analysis, where both labels had exactly the same number of missing observations.

The distribution varies substantially between folders:

- **Folder 0:** 2,138,400 observations are missing labels, representing **17.59%** of the folder's observations.
- **Folder 1:** 14,400 observations are missing labels, representing **0.16%**.
- **Folder 2:** 79,200 observations are missing labels, representing **10.73%**.
- **Folder 3:** 115,200 observations are missing labels, representing **2.33%**.
- **Folder 4:** 1,234,800 observations are missing labels, representing **33.47%**.
- **Folder 5:** 39,600 observations are missing labels, representing **0.30%**.
- **Folder 6:** 21,600 observations are missing labels, representing **0.37%**.
- **Folder 7:** 129,600 observations are missing labels, representing **1.26%**.
- **Folder 8:** 50,400 observations are missing labels, representing **0.72%**.
- **Folder 9:** 205,200 observations are missing labels, representing **2.16%**.

This shows that missing labels are **not uniformly distributed across the event categories**. In particular, Folder 4 has the highest proportion of missing labels at approximately 33.47%, while Folder 1 has only approximately 0.16%.

The fact that `missing_class` and `missing_state` are identical for every folder also indicates that `class` and `state` are missing together in these observations.

Therefore, missing target labels cannot be treated simply as a small, uniformly distributed portion of the dataset. Their distribution across event categories needs to be considered when deciding how these observations will be handled for supervised machine learning.

### Checking Whether Missing Class and State Labels Occur Together

The previous analysis showed that `class` and `state` contain exactly the same number of missing observations across every event category.

This is further examined at the observation level to determine whether the two labels are always missing at the same timestamps.

If both labels are missing together, the missing-label observations form a consistent group. If one label can be present while the other is missing, they would require separate handling during preprocessing.

In [30]:
missing_label_combinations = {
    "both_missing": 0,
    "class_missing_only": 0,
    "state_missing_only": 0,
    "neither_missing": 0
}

for file_path in parquet_files:
    labels = pd.read_parquet(
        file_path,
        columns=["class", "state"]
    )

    class_missing = labels["class"].isna()
    state_missing = labels["state"].isna()

    missing_label_combinations["both_missing"] += (
        class_missing & state_missing
    ).sum()

    missing_label_combinations["class_missing_only"] += (
        class_missing & ~state_missing
    ).sum()

    missing_label_combinations["state_missing_only"] += (
        ~class_missing & state_missing
    ).sum()

    missing_label_combinations["neither_missing"] += (
        ~class_missing & ~state_missing
    ).sum()

missing_label_combinations_df = pd.DataFrame(
    {
        "condition": list(missing_label_combinations.keys()),
        "observations": list(missing_label_combinations.values())
    }
)

missing_label_combinations_df

,condition,observations
0,both_missing,4028400
1,class_missing_only,0
2,state_missing_only,0
3,neither_missing,72558918


### Interpreting the Relationship Between Class and State Missingness

The analysis confirms that `class` and `state` are always missing together.

There are 4,028,400 observations where both labels are missing, while 72,558,918 observations contain values for both labels.

There are no observations where only `class` is missing or only `state` is missing.

Therefore, the missing-label observations form a single consistent group in which both label columns are unavailable simultaneously.

Since `class` is the primary event label for the machine learning task, the next analysis focuses on the actual class values that are present in the dataset. This is necessary to understand the label structure before deciding how the missing-label observations should be handled.

In [31]:
# Calculate the total number of observations directly from the class counts

class_summary = []

total_class_observations = sum(class_counts.values())

for class_value, observations in class_counts.items():
    class_summary.append(
        {
            "class": class_value,
            "observations": observations
        }
    )

class_summary_df = pd.DataFrame(class_summary)

# Calculate the percentage using the total number of class observations

class_summary_df["percentage"] = (
    class_summary_df["observations"]
    / total_class_observations
    * 100
)

# Round percentages to a maximum of 2 decimal places

class_summary_df["percentage"] = (
    class_summary_df["percentage"].round(2)
)

# Sort by class value

class_summary_df = class_summary_df.sort_values(
    "class",
    na_position="first"
)

class_summary_df

,class,observations,percentage
1,<NA>,4028400,5.26
0,0,17305269,22.60
3,1,2909887,3.80
4,2,366858,0.48
6,3,4834079,6.31
7,4,2454883,3.21
8,5,10553279,13.78
10,6,3879083,5.06
13,7,138148,0.18
14,8,744061,0.97


### Investigating the Relationship Between Class Values and Event Categories

The class distribution contains values from two distinct numerical ranges: `0–9` and `101–109`.

The numerical value of a class should not be interpreted as an ordering or magnitude without understanding its meaning in the 3W dataset.

The occurrence of these class values is therefore compared with the folder structure. This determines which class values appear within each event-category folder and whether the special values are associated with particular categories.

This analysis provides the necessary context before deciding how the different class values should be treated as machine learning labels.

In [32]:
folder_class_summary = []

for folder in range(10):
    folder_files = recording_inventory[
        recording_inventory["folder"] == folder
    ]["file_path"]

    class_counts = {}

    for file_path in folder_files:
        labels = pd.read_parquet(
            file_path,
            columns=["class"]
        )

        for class_value, count in labels["class"].value_counts(
            dropna=False
        ).items():
            class_counts[class_value] = (
                class_counts.get(class_value, 0) + count
            )

    for class_value, observations in class_counts.items():
        folder_class_summary.append(
            {
                "folder": folder,
                "class": class_value,
                "observations": observations
            }
        )

folder_class_summary_df = pd.DataFrame(
    folder_class_summary
)

folder_class_summary_df = folder_class_summary_df.sort_values(
    ["folder", "class"],
    na_position="first"
)

folder_class_summary_df

,folder,class,observations
1,0,<NA>,2138400
0,0,0,10019783
5,1,<NA>,14400
3,1,0,916999
4,1,1,2909887
2,1,101,5265821
9,2,<NA>,79200
7,2,0,145036
6,2,2,366858
8,2,102,146691


### Class Values Across Event-Category Folders

The class distribution shows a clear relationship between the numerical `class` values and the folder-based event categories.

The primary event classes `1–9` generally correspond to their respective event-category folders. Special class values `101–109` also occur within the corresponding event folders. For example, folder `1` contains classes `1` and `101`, while folder `7` contains classes `7` and `107`.

Class `0`, representing normal operation, also appears in several event-category folders. This indicates that an event-category folder can contain observations from normal operation as well as the associated event.

Missing class values are also present in every folder, although their proportion varies substantially between folders.

Therefore, the folder structure and the `class` column provide related but different information. The folder should not be treated as the machine learning target without further investigation. In particular, the meaning and temporal behavior of the special class values `101–109` need to be established before defining the final target labels.

### Investigating Special Class Values

The class values `101–109` occur alongside the corresponding event classes within several event-category folders. Their numerical values suggest that they may represent a different labeling state rather than independent event categories.

Their distribution is therefore examined at the file/instance level to determine how frequently they occur and whether they are consistently associated with the corresponding primary event class.

Understanding these values is important because they may need to be treated differently from the primary event classes when constructing the machine learning target.

In [33]:
special_class_summary = []

special_classes = [101, 102, 105, 106, 107, 108, 109]

for file_path in parquet_files:
    labels = pd.read_parquet(
        file_path,
        columns=["class"]
    )

    class_counts = labels["class"].value_counts(
        dropna=False
    )

    for class_value in special_classes:
        observations = class_counts.get(class_value, 0)

        if observations > 0:
            special_class_summary.append(
                {
                    "recording_id": file_path.stem,
                    "folder": int(file_path.parent.name),
                    "class": class_value,
                    "observations": observations
                }
            )

special_class_summary_df = pd.DataFrame(
    special_class_summary
)

special_class_summary_df = special_class_summary_df.sort_values(
    ["folder", "class", "recording_id"]
)

special_class_summary_df.head(20)

,recording_id,folder,class,observations
0,DRAWN_00001,1,101,148616
1,DRAWN_00002,1,101,74308
2,DRAWN_00003,1,101,19017
3,DRAWN_00004,1,101,28508
4,DRAWN_00005,1,101,12959
5,DRAWN_00006,1,101,53570
6,DRAWN_00007,1,101,41478
7,DRAWN_00008,1,101,11529
8,DRAWN_00009,1,101,18155
9,DRAWN_00010,1,101,25941


### Examining Class Transitions Within Files

The previous analysis established that special class values such as `101–109` occur within the same event-category folders as their corresponding primary event classes. Their meaning cannot be determined from frequency alone.

The temporal order of class values within each file/instance is therefore examined. Class transitions are counted in chronological order, with particular attention to transitions involving the special values `101–109`.

This helps determine whether the special classes occur as transitions between normal and event states, alongside the primary event class, or as distinct periods within an instance.

### Class Transitions Within Event-Category Folders

Class transitions are examined separately for each folder because each folder represents a specific event category.

This preserves the relationship between the folder and its associated event classes while showing how class values change chronologically within individual files/instances.

The analysis focuses particularly on transitions involving the special class values `101–109`. Comparing these transitions within their corresponding folders helps determine whether the special values consistently occur between normal operation and the associated event class, or represent another stage of the event labeling.

In [34]:
folder_transition_summary = []

for file_path in parquet_files:
    labels = pd.read_parquet(
        file_path,
        columns=["class"]
    )

    class_values = labels["class"].tolist()

    previous_class = class_values[0]

    for current_class in class_values[1:]:

        # Ignore consecutive missing class values.
        if pd.isna(current_class) and pd.isna(previous_class):
            previous_class = current_class
            continue

        # A missing-to-valid or valid-to-missing change is a transition.
        if pd.isna(current_class) or pd.isna(previous_class):
            class_changed = True
        else:
            class_changed = current_class != previous_class

        if class_changed:
            folder_transition_summary.append(
                {
                    "folder": int(file_path.parent.name),
                    "from_class": previous_class,
                    "to_class": current_class
                }
            )

        previous_class = current_class

folder_transition_df = pd.DataFrame(
    folder_transition_summary
)

folder_transition_counts = (
    folder_transition_df
    .groupby(
        ["folder", "from_class", "to_class"],
        dropna=False
    )
    .size()
    .reset_index(name="transitions")
    .sort_values(
        ["folder", "transitions"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

folder_transition_counts

,folder,from_class,to_class,transitions
0,0,NaN,0,594
1,1,0.0,101,128
2,1,101.0,1,124
3,1,NaN,0,4
4,2,0.0,102,38
5,2,102.0,2,27
6,2,NaN,0,22
7,3,NaN,3,32
8,4,NaN,4,343
9,5,0.0,105,450


### Measuring the Duration of Special Class Phases

The transition analysis shows that special classes such as `101–109` occur between normal operation and their corresponding event classes in many files/instances.

The duration of these special-class periods is therefore examined to determine whether they represent substantial portions of the event timeline or only short transition periods.

For each file/instance containing a special class, the number of observations assigned to that class is compared with the total number of observations in the file/instance. Since each observation corresponds to one row in the time series, this provides a direct measure of the duration of the special-class phase.

In [35]:
special_class_duration = []

for file_path in parquet_files:
    labels = pd.read_parquet(
        file_path,
        columns=["class"]
    )

    total_observations = len(labels)

    for special_class in special_classes:
        special_observations = (
            labels["class"] == special_class
        ).sum()

        if special_observations > 0:
            special_class_duration.append(
                {
                    "recording_id": file_path.stem,
                    "folder": int(file_path.parent.name),
                    "class": special_class,
                    "special_class_observations": special_observations,
                    "total_observations": total_observations,
                    "percentage_of_file": (
                        special_observations
                        / total_observations
                        * 100
                    )
                }
            )

special_class_duration_df = pd.DataFrame(
    special_class_duration
)

special_class_duration_df["percentage_of_file"] = (
    special_class_duration_df["percentage_of_file"].round(2)
)

special_class_duration_df = special_class_duration_df.sort_values(
    ["folder", "class", "percentage_of_file"],
    ascending=[True, True, False]
).reset_index(drop=True)

special_class_duration_df.head(20)

,recording_id,folder,class,special_class_observations,total_observations,percentage_of_file
0,DRAWN_00007,1,101,41478,43200,96.01
1,DRAWN_00005,1,101,12959,14400,89.99
2,DRAWN_00003,1,101,19017,21600,88.04
3,DRAWN_00001,1,101,148616,172800,86.00
4,DRAWN_00002,1,101,74308,86400,86.00
5,DRAWN_00009,1,101,18155,21600,84.05
6,DRAWN_00008,1,101,11529,14400,80.06
7,DRAWN_00004,1,101,28508,43200,65.99
8,SIMULATED_00007,1,101,57600,88799,64.87
9,SIMULATED_00010,1,101,57600,88799,64.87


### Comparing Special Classes With Their Corresponding Event Classes

The special classes occupy substantial portions of many files/instances, so they cannot be treated as simple transition points.

The next analysis compares each special class with its corresponding primary event class within the same file/instance. For each file/instance, the number and proportion of observations assigned to both classes are calculated.

This comparison determines whether the special class and primary event class typically form separate phases within the same instance and provides a clearer basis for understanding their relationship before target construction.

In [36]:
special_vs_event_summary = []

special_to_event = {
    101: 1,
    102: 2,
    105: 5,
    106: 6,
    107: 7,
    108: 8,
    109: 9
}

for file_path in parquet_files:
    labels = pd.read_parquet(
        file_path,
        columns=["class"]
    )

    total_observations = len(labels)

    for special_class, event_class in special_to_event.items():

        special_observations = (
            labels["class"] == special_class
        ).sum()

        event_observations = (
            labels["class"] == event_class
        ).sum()

        if special_observations > 0 or event_observations > 0:
            special_vs_event_summary.append(
                {
                    "recording_id": file_path.stem,
                    "folder": int(file_path.parent.name),
                    "special_class": special_class,
                    "event_class": event_class,
                    "special_observations": special_observations,
                    "event_observations": event_observations,
                    "special_percentage": (
                        special_observations
                        / total_observations
                        * 100
                    ),
                    "event_percentage": (
                        event_observations
                        / total_observations
                        * 100
                    )
                }
            )

special_vs_event_df = pd.DataFrame(
    special_vs_event_summary
)

special_vs_event_df[
    "special_percentage"
] = special_vs_event_df[
    "special_percentage"
].round(2)

special_vs_event_df[
    "event_percentage"
] = special_vs_event_df[
    "event_percentage"
].round(2)

special_vs_event_df = special_vs_event_df.sort_values(
    ["folder", "special_class", "special_percentage"],
    ascending=[True, True, False]
).reset_index(drop=True)

special_vs_event_df.head(20)

,recording_id,folder,special_class,event_class,special_observations,event_observations,special_percentage,event_percentage
0,DRAWN_00007,1,101,1,41478,0,96.01,0.00
1,DRAWN_00005,1,101,1,12959,0,89.99,0.00
2,DRAWN_00003,1,101,1,19017,422,88.04,1.95
3,DRAWN_00001,1,101,1,148616,10405,86.00,6.02
4,DRAWN_00002,1,101,1,74308,5203,86.00,6.02
5,DRAWN_00009,1,101,1,18155,2162,84.05,10.01
6,DRAWN_00008,1,101,1,11529,1430,80.06,9.93
7,DRAWN_00004,1,101,1,28508,0,65.99,0.00
8,SIMULATED_00007,1,101,1,57600,25199,64.87,28.38
9,SIMULATED_00010,1,101,1,57600,25199,64.87,28.38


### Comparing Special Classes With Their Corresponding Event Classes

The special classes occupy a large part of many files/instances. They are therefore not simply short transition periods.

For example, in `DRAWN_00001`, class `101` represents 86.00% of the observations, while class `1` represents 6.02%. The class sequence in this type of instance is:

`0 → 101 → 1`

This shows that class `101` can represent a long period before the corresponding event class `1` appears.

Some files/instances contain the special class without containing the corresponding event class. For example, `DRAWN_00007` contains class `101` for 96.01% of its observations but contains no class `1`.

Therefore, the special classes `101–109` should not yet be treated as simple transition labels or automatically merged with classes `1–9`. Their exact meaning needs to be established before defining the machine learning target.

### Comparing Special Classes With Their Corresponding Event Classes

The special classes occupy a large part of many files/instances. They are therefore not simply short transition periods.

For example, in `DRAWN_00001`, class `101` represents 86.00% of the observations, while class `1` represents 6.02%. The class sequence in this type of instance is:

`0 → 101 → 1`

This shows that class `101` can represent a long period before the corresponding event class `1` appears.

Some files/instances contain the special class without containing the corresponding event class. For example, `DRAWN_00007` contains class `101` for 96.01% of its observations but contains no class `1`.

Therefore, the special classes `101–109` should not yet be treated as simple transition labels or automatically merged with classes `1–9`. Their exact meaning needs to be established before defining the machine learning target.

### Verifying Timestamp Order and Sampling Gaps

The 3W dataset contains time-ordered observations within each file/instance. Before applying further preprocessing, the timestamp index is checked across the complete dataset.

The analysis verifies whether timestamps are ordered correctly and identifies gaps in the expected time sequence. This is important because missing timestamps represent a different situation from missing sensor values.

Timestamp gaps are evaluated separately for each file/instance so that independent files are not incorrectly treated as one continuous time series.

In [37]:
timestamp_quality_summary = []

for file_path in parquet_files:
    timestamps = pd.read_parquet(
        file_path,
        columns=[]  # Read only the timestamp index
    ).index

    is_sorted = timestamps.is_monotonic_increasing

    time_differences = timestamps.to_series().diff().dropna()

    one_second_intervals = (
        time_differences == pd.Timedelta(seconds=1)
    ).sum()

    timestamp_gaps = (
        time_differences > pd.Timedelta(seconds=1)
    ).sum()

    largest_gap = (
        time_differences.max()
        if len(time_differences) > 0
        else pd.Timedelta(0)
    )

    timestamp_quality_summary.append(
        {
            "recording_id": file_path.stem,
            "folder": int(file_path.parent.name),
            "observations": len(timestamps),
            "timestamps_sorted": is_sorted,
            "one_second_intervals": one_second_intervals,
            "timestamp_gaps": timestamp_gaps,
            "largest_gap": largest_gap
        }
    )

timestamp_quality_df = pd.DataFrame(
    timestamp_quality_summary
)

timestamp_quality_df.head()

,recording_id,folder,observations,timestamps_sorted,one_second_intervals,timestamp_gaps,largest_gap
0,WELL-00001_20170201010207,0,21474,True,21473,0,0 days 00:00:01
1,WELL-00001_20170201060114,0,21527,True,21526,0,0 days 00:00:01
2,WELL-00001_20170201110124,0,21517,True,21516,0,0 days 00:00:01
3,WELL-00001_20170201160311,0,21410,True,21409,0,0 days 00:00:01
4,WELL-00001_20170201210228,0,21453,True,21452,0,0 days 00:00:01


### Dataset-Wide Timestamp Quality

The initial timestamp check shows correctly ordered observations with one-second sampling and no gaps in the inspected files.

The timestamp quality is now summarized across all files/instances to determine whether this behavior is consistent throughout the complete dataset.

The summary counts files/instances with correctly ordered timestamps, files/instances containing gaps larger than one second, and the maximum timestamp gap observed anywhere in the dataset.

In [38]:
timestamp_quality_overview = {
    "total_files": len(timestamp_quality_df),
    "sorted_files": timestamp_quality_df[
        "timestamps_sorted"
    ].sum(),
    "files_with_timestamp_gaps": (
        timestamp_quality_df["timestamp_gaps"] > 0
    ).sum(),
    "maximum_timestamp_gap": timestamp_quality_df[
        "largest_gap"
    ].max()
}

timestamp_quality_overview

{'total_files': 2228,
 'sorted_files': np.int64(2228),
 'files_with_timestamp_gaps': np.int64(0),
 'maximum_timestamp_gap': Timedelta('0 days 00:00:01')}

### Identifying Continuous and Status Sensors

The sensor variables have different types of measurements. Some represent continuous physical quantities such as pressure, temperature, and flow, while others represent operating states or equipment conditions.

The storage data type alone is not sufficient to determine the sensor type because status variables may also be stored as numerical values.

The number of unique non-missing values and the observed value patterns are therefore examined for each retained sensor. This helps distinguish continuous measurements from discrete status variables before selecting the appropriate preprocessing strategy.

In [39]:
sensor_type_summary = []

for sensor in available_sensors:
    unique_values = set()
    total_non_missing = 0

    for file_path in parquet_files:
        sensor_data = pd.read_parquet(
            file_path,
            columns=[sensor]
        )[sensor].dropna()

        if len(sensor_data) > 0:
            unique_values.update(
                sensor_data.unique().tolist()
            )
            total_non_missing += len(sensor_data)

    sensor_type_summary.append(
        {
            "sensor": sensor,
            "unique_values": len(unique_values),
            "non_missing_observations": total_non_missing
        }
    )

sensor_type_summary_df = pd.DataFrame(
    sensor_type_summary
)

sensor_type_summary_df = sensor_type_summary_df.sort_values(
    "unique_values"
)

sensor_type_summary_df

,sensor,unique_values,non_missing_observations
3,ESTADO-M1,2,17773205
2,ESTADO-DHSV,2,13865397
6,ESTADO-SDV-GL,2,17105271
5,ESTADO-PXO,2,18176203
7,ESTADO-SDV-P,2,24316107
8,ESTADO-W1,2,18938036
10,ESTADO-XO,2,18538730
4,ESTADO-M2,3,17688282
9,ESTADO-W2,3,18293462
14,P-MON-CKGL,92596,308726


### Understanding Sensor Types From Unique Values

The number of unique values shows that the sensors are not all the same type.

The `ESTADO-*` sensors have only **2 or 3 unique values**. These are discrete status variables that represent operating states rather than continuous physical measurements.

The remaining sensors have thousands or millions of unique values. These behave like continuous measurements such as pressure, temperature, flow, or other physical quantities.

Therefore, the sensors can be broadly separated into two groups:

- **Status/discrete sensors:** `ESTADO-M1`, `ESTADO-DHSV`, `ESTADO-SDV-GL`, `ESTADO-PXO`, `ESTADO-SDV-P`, `ESTADO-W1`, `ESTADO-XO`, `ESTADO-M2`, and `ESTADO-W2`.
- **Continuous sensors:** the remaining 14 retained sensors.

This distinction is important because missing values should not automatically be handled in the same way for status and continuous sensors.

### Establishing the Missing-Value Strategy

The previous analysis identified two broad sensor types: continuous measurements and discrete status variables.

Missingness is substantial and varies considerably between sensors and files/instances. Some sensors are completely unavailable in individual files/instances, while others contain intermittent missing observations.

Because of these differences, a single imputation method should not be applied blindly to every sensor. The preprocessing strategy must preserve the meaning of each sensor and avoid creating artificial values where no reliable information exists.

The next step is to summarize missingness by sensor type and identify how often each sensor is completely unavailable within a file/instance.

In [40]:
sensor_missing_strategy_summary = []

status_sensors = [
    "ESTADO-M1",
    "ESTADO-DHSV",
    "ESTADO-SDV-GL",
    "ESTADO-PXO",
    "ESTADO-SDV-P",
    "ESTADO-W1",
    "ESTADO-XO",
    "ESTADO-M2",
    "ESTADO-W2"
]

continuous_sensors = [
    sensor
    for sensor in available_sensors
    if sensor not in status_sensors
]

for sensor in available_sensors:
    completely_missing_files = 0
    partially_missing_files = 0
    complete_files = 0

    for file_path in parquet_files:
        sensor_data = pd.read_parquet(
            file_path,
            columns=[sensor]
        )[sensor]

        missing_count = sensor_data.isna().sum()

        if missing_count == len(sensor_data):
            completely_missing_files += 1
        elif missing_count > 0:
            partially_missing_files += 1
        else:
            complete_files += 1

    sensor_type = (
        "status/discrete"
        if sensor in status_sensors
        else "continuous"
    )

    sensor_missing_strategy_summary.append(
        {
            "sensor": sensor,
            "sensor_type": sensor_type,
            "completely_missing_files": completely_missing_files,
            "partially_missing_files": partially_missing_files,
            "complete_files": complete_files
        }
    )

sensor_missing_strategy_df = pd.DataFrame(
    sensor_missing_strategy_summary
)

sensor_missing_strategy_df = sensor_missing_strategy_df.sort_values(
    ["sensor_type", "completely_missing_files"],
    ascending=[True, False]
).reset_index(drop=True)

sensor_missing_strategy_df

,sensor,sensor_type,completely_missing_files,partially_missing_files,complete_files
0,P-MON-CKGL,continuous,2226,1,1
1,ABER-CKGL,continuous,1942,16,270
2,ABER-CKP,continuous,1929,19,280
3,T-PDG,continuous,1665,40,523
4,QGL,continuous,1633,28,567
5,P-JUS-CKP,continuous,1563,3,662
6,T-MON-CKP,continuous,1559,5,664
7,P-JUS-CKGL,continuous,1389,34,805
8,P-ANULAR,continuous,1307,49,872
9,T-JUS-CKP,continuous,557,27,1644


### Selecting the Common Sensor Set

The missingness analysis shows that several sensors are unavailable in most files/instances. Including these sensors would require extensive imputation and could introduce unreliable information into the model.

Sensors with very low file-level coverage are therefore excluded from the common feature set. Sensors with sufficient coverage are retained because their missing values can be handled within files/instances where the sensor is available.

This creates a consistent set of sensor features while avoiding dependence on sensors that are rarely available.

In [41]:
# Calculate the number and percentage of files/instances
# in which each sensor has at least some valid data.

sensor_coverage_summary = sensor_missing_strategy_df.copy()

sensor_coverage_summary["files_with_data"] = (
    sensor_coverage_summary["complete_files"]
    + sensor_coverage_summary["partially_missing_files"]
)

sensor_coverage_summary["coverage_percentage"] = (
    sensor_coverage_summary["files_with_data"]
    / len(parquet_files)
    * 100
)

sensor_coverage_summary["coverage_percentage"] = (
    sensor_coverage_summary["coverage_percentage"].round(2)
)

sensor_coverage_summary = sensor_coverage_summary.sort_values(
    "coverage_percentage",
    ascending=False
).reset_index(drop=True)


# Retain sensors that are available in at least 25% of files/instances.

minimum_coverage = 25.0

selected_sensors = sensor_coverage_summary[
    sensor_coverage_summary["coverage_percentage"] >= minimum_coverage
]["sensor"].tolist()

excluded_sensors = sensor_coverage_summary[
    sensor_coverage_summary["coverage_percentage"] < minimum_coverage
]["sensor"].tolist()

print("Selected sensors:")
print(selected_sensors)

print("\nExcluded sensors:")
print(excluded_sensors)

print("\nNumber of selected sensors:", len(selected_sensors))
print("Number of excluded sensors:", len(excluded_sensors))

Selected sensors:
['P-TPT', 'P-PDG', 'T-TPT', 'P-MON-CKP', 'T-JUS-CKP', 'P-ANULAR', 'P-JUS-CKGL', 'T-MON-CKP', 'P-JUS-CKP', 'ESTADO-SDV-P', 'QGL', 'T-PDG', 'ESTADO-W1']

Excluded sensors:
['ESTADO-XO', 'ESTADO-W2', 'ESTADO-PXO', 'ESTADO-M2', 'ESTADO-M1', 'ESTADO-SDV-GL', 'ESTADO-DHSV', 'ABER-CKP', 'ABER-CKGL', 'P-MON-CKGL']

Number of selected sensors: 13
Number of excluded sensors: 10


### Understanding File-Level Sensor Coverage

The file-level coverage calculation considers a sensor to be available if **at least one valid observation exists** for that sensor within a file/instance.

For example, if a file contains 100,000 observations:

- 0 valid observations → sensor is considered unavailable.
- 1 valid observation → sensor is considered available.
- 50,000 valid observations → sensor is considered available.
- 100,000 valid observations → sensor is considered available.

Therefore, the 25% coverage threshold means that a sensor must appear in at least 25% of the files/instances, but it does **not** mean that the sensor has valid values for 25% of all observations.

This distinction is important because a sensor could be counted as available in a file even if almost all of its observations are missing. Therefore, file-level coverage alone should not be used to make the final sensor-selection decision.

### Comparing File-Level and Observation-Level Sensor Coverage

File-level coverage only indicates whether a sensor appears at least once in a file/instance. It does not show how much of that file contains valid sensor observations.

Observation-level coverage is therefore calculated for each sensor across the complete dataset. This provides a second measure of data availability and helps identify sensors that may appear in many files but contain valid values for only a small proportion of observations.

Both measures are considered before finalizing the common sensor set.

In [42]:
observation_coverage_summary = []

total_dataset_observations = len(parquet_files)

for sensor in available_sensors:
    total_observations = 0
    valid_observations = 0

    for file_path in parquet_files:
        sensor_data = pd.read_parquet(
            file_path,
            columns=[sensor]
        )[sensor]

        total_observations += len(sensor_data)
        valid_observations += sensor_data.notna().sum()

    observation_coverage_summary.append(
        {
            "sensor": sensor,
            "valid_observations": valid_observations,
            "total_observations": total_observations,
            "observation_coverage_percentage": (
                valid_observations
                / total_observations
                * 100
            )
        }
    )

observation_coverage_df = pd.DataFrame(
    observation_coverage_summary
)

observation_coverage_df[
    "observation_coverage_percentage"
] = (
    observation_coverage_df[
        "observation_coverage_percentage"
    ].round(2)
)

observation_coverage_df = observation_coverage_df.sort_values(
    "observation_coverage_percentage",
    ascending=False
).reset_index(drop=True)

observation_coverage_df

,sensor,valid_observations,total_observations,observation_coverage_percentage
0,P-TPT,71205535,76587318,92.97
1,P-MON-CKP,69507467,76587318,90.76
2,P-PDG,68671102,76587318,89.66
3,T-TPT,66325278,76587318,86.60
4,T-JUS-CKP,60554418,76587318,79.07
5,P-JUS-CKGL,31896234,76587318,41.65
6,P-ANULAR,25460515,76587318,33.24
7,QGL,24565960,76587318,32.08
8,ESTADO-SDV-P,24316107,76587318,31.75
9,T-MON-CKP,23617605,76587318,30.84


### Comparing Sensor Coverage Measures

The file-level and observation-level coverage measures provide different views of sensor availability.

File-level coverage measures how many files/instances contain at least one valid value for a sensor. Observation-level coverage measures the proportion of all observations for which the sensor contains a valid value.

These measures can produce different results. For example, `ESTADO-W1` appears in enough files to pass the 25% file-level threshold, but only 24.73% of all observations contain a valid value.

The two measures are therefore compared before selecting the final sensor set. This avoids selecting a sensor based only on minimal availability in individual files/instances.

In [43]:
# Combine file-level and observation-level coverage
# into one summary for comparison.

sensor_coverage_comparison = sensor_coverage_summary[
    [
        "sensor",
        "sensor_type",
        "files_with_data",
        "coverage_percentage"
    ]
].rename(
    columns={
        "coverage_percentage": "file_coverage_percentage"
    }
)

sensor_coverage_comparison = sensor_coverage_comparison.merge(
    observation_coverage_df[
        [
            "sensor",
            "observation_coverage_percentage"
        ]
    ],
    on="sensor",
    how="left"
)

sensor_coverage_comparison = sensor_coverage_comparison.sort_values(
    "observation_coverage_percentage",
    ascending=False
).reset_index(drop=True)

sensor_coverage_comparison

,sensor,sensor_type,files_with_data,file_coverage_percentage,observation_coverage_percentage
0,P-TPT,continuous,2069,92.86,92.97
1,P-MON-CKP,continuous,1831,82.18,90.76
2,P-PDG,continuous,1936,86.89,89.66
3,T-TPT,continuous,1929,86.58,86.60
4,T-JUS-CKP,continuous,1671,75.00,79.07
5,P-JUS-CKGL,continuous,839,37.66,41.65
6,P-ANULAR,continuous,921,41.34,33.24
7,QGL,continuous,595,26.71,32.08
8,ESTADO-SDV-P,status/discrete,661,29.67,31.75
9,T-MON-CKP,continuous,669,30.03,30.84


### Defining the Final Sensor Coverage Criterion

Both file-level and observation-level coverage are considered when selecting the common sensor set.

A sensor should provide information across a meaningful portion of the dataset and should not be available only in a very small number of files/instances.

For this preprocessing stage, a sensor is retained when it has:

- at least **25% file-level coverage**, and
- at least **25% observation-level coverage**.

This criterion removes sensors with very limited availability while retaining sensors that provide substantial information across the dataset.

In [44]:
# Define the final sensor coverage threshold.

minimum_file_coverage = 25.0
minimum_observation_coverage = 25.0

final_sensor_coverage = sensor_coverage_comparison[
    (
        sensor_coverage_comparison["file_coverage_percentage"]
        >= minimum_file_coverage
    )
    &
    (
        sensor_coverage_comparison[
            "observation_coverage_percentage"
        ]
        >= minimum_observation_coverage
    )
].copy()

final_selected_sensors = final_sensor_coverage[
    "sensor"
].tolist()

final_excluded_sensors = sensor_coverage_comparison[
    ~sensor_coverage_comparison["sensor"].isin(
        final_selected_sensors
    )
]["sensor"].tolist()

print("Final selected sensors:")
print(final_selected_sensors)

print("\nFinal excluded sensors:")
print(final_excluded_sensors)

print("\nNumber of selected sensors:", len(final_selected_sensors))
print("Number of excluded sensors:", len(final_excluded_sensors))

Final selected sensors:
['P-TPT', 'P-MON-CKP', 'P-PDG', 'T-TPT', 'T-JUS-CKP', 'P-JUS-CKGL', 'P-ANULAR', 'QGL', 'ESTADO-SDV-P', 'T-MON-CKP', 'P-JUS-CKP', 'T-PDG']

Final excluded sensors:
['ESTADO-W1', 'ESTADO-XO', 'ESTADO-W2', 'ESTADO-PXO', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-SDV-GL', 'ESTADO-DHSV', 'ABER-CKP', 'ABER-CKGL', 'P-MON-CKGL']

Number of selected sensors: 12
Number of excluded sensors: 11


### Selecting the Missing-Value Treatment

The missing-value analysis of the selected sensors is used to define the preprocessing treatment.

Short internal gaps in continuous sensors can be handled using time-based interpolation when sufficient observations exist on both sides of the gap. Longer gaps are not automatically interpolated because doing so could create a large amount of artificial data.

For status/discrete sensors, interpolation is avoided because intermediate numerical values may not represent valid operating states.

The preprocessing therefore distinguishes between continuous and status/discrete sensors and between short and long missing gaps.

In [45]:
# Recalculate missing-block information for the 12 selected sensors.

selected_sensor_missing_blocks = []

for file_path in parquet_files:
    recording = pd.read_parquet(
        file_path,
        columns=final_selected_sensors
    )

    for sensor in final_selected_sensors:
        is_missing = recording[sensor].isna()

        if not is_missing.any():
            continue

        groups = (
            is_missing.ne(is_missing.shift())
            .cumsum()
        )

        block_lengths = (
            is_missing
            .groupby(groups)
            .sum()
        )

        missing_block_lengths = block_lengths[
            block_lengths > 0
        ]

        selected_sensor_missing_blocks.append(
            {
                "recording_id": file_path.stem,
                "folder": int(file_path.parent.name),
                "sensor": sensor,
                "number_of_missing_blocks": len(
                    missing_block_lengths
                ),
                "total_missing_observations": (
                    missing_block_lengths.sum()
                ),
                "longest_missing_block": (
                    missing_block_lengths.max()
                )
            }
        )

selected_sensor_missing_blocks_df = pd.DataFrame(
    selected_sensor_missing_blocks
)

missing_block_summary = (
    selected_sensor_missing_blocks_df
    .groupby("sensor")
    .agg(
        files_with_missing_data=("recording_id", "nunique"),
        total_missing_observations=(
            "total_missing_observations",
            "sum"
        ),
        maximum_missing_block=(
            "longest_missing_block",
            "max"
        )
    )
    .reset_index()
)

missing_block_summary = missing_block_summary.sort_values(
    "maximum_missing_block",
    ascending=False
).reset_index(drop=True)

missing_block_summary

,sensor,files_with_missing_data,total_missing_observations,maximum_missing_block
0,P-JUS-CKP,1271,54471358,768314
1,QGL,1151,52021358,768314
2,T-MON-CKP,1269,52969713,768314
3,P-JUS-CKGL,913,44691084,722192
4,T-JUS-CKP,584,16032900,722192
5,P-ANULAR,696,51126803,680420
6,T-PDG,1045,55776046,680420
7,P-PDG,341,7916216,680420
8,T-TPT,350,10262040,586201
9,P-TPT,212,5381783,586201


### What Each Column Means

- **`sensor`** — the sensor being analyzed.
- **`files_with_missing_data`** — number of the 2,228 files/instances in which that sensor has at least one missing observation.
- **`total_missing_observations`** — total number of missing rows for that sensor across the entire dataset.
- **`maximum_missing_block`** — the largest continuous sequence of missing observations found for that sensor in any single file.

### Distinguishing Short and Long Missing Gaps

The selected sensors contain both short missing gaps and long periods of sensor unavailability.

To guide the preprocessing strategy, missing blocks are grouped into three categories:

- **Short gaps:** fewer than 60 consecutive missing observations.
- **Medium gaps:** 60 to 3,600 consecutive missing observations.
- **Long gaps:** more than 3,600 consecutive missing observations.

This classification helps determine whether a missing value is likely to be an isolated data gap or part of a longer period in which the sensor was unavailable.

In [46]:
# Classify missing blocks by their length.

missing_block_categories = []

for _, row in selected_sensor_missing_blocks_df.iterrows():
    block_length = row["longest_missing_block"]

    if block_length < 60:
        category = "short"
    elif block_length <= 3600:
        category = "medium"
    else:
        category = "long"

    missing_block_categories.append(category)

selected_sensor_missing_blocks_df["missing_block_category"] = (
    missing_block_categories
)

missing_block_category_summary = (
    selected_sensor_missing_blocks_df
    .groupby(["sensor", "missing_block_category"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

missing_block_category_summary

missing_block_category,sensor,long,medium,short
0,ESTADO-SDV-P,1567,23,17
1,P-ANULAR,1307,31,18
2,P-JUS-CKGL,1389,14,20
3,P-JUS-CKP,1563,2,1
4,P-MON-CKP,397,14,22
5,P-PDG,301,25,15
6,P-TPT,159,32,21
7,QGL,1633,17,11
8,T-JUS-CKP,557,12,15
9,T-MON-CKP,1559,3,2


### Choosing a Missing-Value Strategy

The missing-value analysis shows that long gaps dominate the selected sensor set. These gaps often represent periods in which a sensor is unavailable rather than isolated missing observations.

Therefore, missing values will not be filled indiscriminately.

- **Short gaps** may be interpolated when appropriate for continuous sensors.
- **Medium and long gaps** will remain missing rather than being artificially filled across extended periods of sensor unavailability.
- **Status/discrete sensors** will not be interpolated because their values represent discrete operating states.

This approach preserves the distinction between an actual sensor measurement and a period in which the sensor was unavailable.

In [47]:
# Identify the selected continuous and status/discrete sensors.

selected_status_sensors = [
    sensor
    for sensor in final_selected_sensors
    if sensor in status_sensors
]

selected_continuous_sensors = [
    sensor
    for sensor in final_selected_sensors
    if sensor not in status_sensors
]

print("Selected continuous sensors:")
print(selected_continuous_sensors)

print("\nSelected status/discrete sensors:")
print(selected_status_sensors)

Selected continuous sensors:
['P-TPT', 'P-MON-CKP', 'P-PDG', 'T-TPT', 'T-JUS-CKP', 'P-JUS-CKGL', 'P-ANULAR', 'QGL', 'T-MON-CKP', 'P-JUS-CKP', 'T-PDG']

Selected status/discrete sensors:
['ESTADO-SDV-P']


### Defining the Short-Gap Imputation Rule

Short missing gaps will be handled differently from medium and long gaps.

For the 11 continuous sensors, only short internal gaps of fewer than 60 consecutive observations will be considered for interpolation.

Interpolation will be performed within each file/instance using the surrounding observed values. Missing gaps at the beginning or end of a file will not be extrapolated.

The `ESTADO-SDV-P` status sensor will not be interpolated because its values represent discrete operating states.

In [48]:
# Define the maximum gap length that can be interpolated.

maximum_interpolation_gap = 59

print(
    "Maximum missing-gap length eligible for interpolation:",
    maximum_interpolation_gap,
    "observations"
)

Maximum missing-gap length eligible for interpolation: 59 observations


### Applying Short-Gap Interpolation

Short internal gaps in the 11 continuous sensors are interpolated only when the missing block contains fewer than 60 consecutive observations.

Interpolation is performed independently within each file/instance so that observations from different files are never combined.

Missing values at the beginning or end of a file are not extrapolated. Medium and long missing blocks remain missing because filling extended periods of sensor unavailability would introduce artificial measurements.

The `ESTADO-SDV-P` status sensor remains unchanged.

In [49]:
# Define the final columns used in the processed dataset.

final_columns = (
    final_selected_sensors
    + ["class", "state"]
)

print("Final columns:")
print(final_columns)

Final columns:
['P-TPT', 'P-MON-CKP', 'P-PDG', 'T-TPT', 'T-JUS-CKP', 'P-JUS-CKGL', 'P-ANULAR', 'QGL', 'ESTADO-SDV-P', 'T-MON-CKP', 'P-JUS-CKP', 'T-PDG', 'class', 'state']


In [50]:
# Create the folder where processed Parquet files will be saved.
processed_path = project_path / "data" / "processed" / "3W"

processed_path.mkdir(
    parents=True,
    exist_ok=True
)


# Store only the small interpolation summary in memory.
interpolation_summary = []


# Process each Parquet file separately.
# Each file is processed, saved, and then released from memory.
for file_path in parquet_files:

    # Read only the selected sensor features and the class/state labels.
    recording = pd.read_parquet(
        file_path,
        columns=final_columns
    )

    # Process each of the 11 continuous sensors.
    for sensor in selected_continuous_sensors:

        # Count missing values before interpolation.
        missing_before = recording[sensor].isna().sum()

        # Fill only short internal missing gaps using linear interpolation.
        recording[sensor] = (
            recording[sensor]
            .interpolate(
                method="linear",
                limit=maximum_interpolation_gap,
                limit_area="inside"
            )
        )

        # Count missing values after interpolation.
        missing_after = recording[sensor].isna().sum()

        # Store how many values were actually interpolated.
        interpolation_summary.append(
            {
                "recording_id": file_path.stem,
                "sensor": sensor,
                "values_interpolated": (
                    missing_before - missing_after
                )
            }
        )

    # Create the corresponding folder for this processed file.
    output_folder = processed_path / file_path.parent.name
    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    # Save the processed file immediately instead of keeping it in memory.
    output_file = output_folder / file_path.name
    recording.to_parquet(output_file)

    # Delete the current file from memory before moving to the next file.
    del recording


# Convert the interpolation results into a DataFrame.
interpolation_summary_df = pd.DataFrame(
    interpolation_summary
)

# Display the first few rows of the interpolation summary.
interpolation_summary_df.head()

,recording_id,sensor,values_interpolated
0,WELL-00001_20170201010207,P-TPT,0
1,WELL-00001_20170201010207,P-MON-CKP,0
2,WELL-00001_20170201010207,P-PDG,0
3,WELL-00001_20170201010207,T-TPT,0
4,WELL-00001_20170201010207,T-JUS-CKP,0


### Reviewing the Interpolation Results

The interpolation summary records the number of values filled for each sensor in each file/instance.

The total number of interpolated observations is examined to verify that the short-gap interpolation step actually modified the dataset.

The remaining missing values will be preserved for longer gaps and periods of sensor unavailability.

In [51]:
# Summarize the total number of values interpolated for each sensor.

interpolation_totals = (
    interpolation_summary_df
    .groupby("sensor")["values_interpolated"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

interpolation_totals

,sensor,values_interpolated
0,P-TPT,3651
1,P-ANULAR,3604
2,T-TPT,3420
3,P-PDG,3112
4,T-PDG,2188
5,P-MON-CKP,1969
6,P-JUS-CKGL,1874
7,QGL,1648
8,T-JUS-CKP,1602
9,T-MON-CKP,482


### Validating Remaining Missing Values

After short-gap interpolation, the processed files are checked to confirm that missing values remain only where they were intentionally preserved.

This validation verifies that:

- short internal gaps eligible for interpolation were filled,
- longer missing gaps remain missing,
- missing values at file boundaries remain unchanged, and
- the `ESTADO-SDV-P` status sensor was not interpolated.

In [52]:
# Check the remaining missing values in the processed dataset.

remaining_missing_summary = []

processed_files = sorted(
    processed_path.glob("*/*.parquet")
)

for file_path in processed_files:

    recording = pd.read_parquet(
        file_path,
        columns=final_columns
    )

    for sensor in final_selected_sensors:

        missing_count = recording[sensor].isna().sum()

        if missing_count > 0:
            remaining_missing_summary.append(
                {
                    "recording_id": file_path.stem,
                    "sensor": sensor,
                    "missing_observations": missing_count
                }
            )

remaining_missing_df = pd.DataFrame(
    remaining_missing_summary
)

remaining_missing_by_sensor = (
    remaining_missing_df
    .groupby("sensor")["missing_observations"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

remaining_missing_by_sensor

,sensor,missing_observations
0,T-PDG,55773858
1,P-JUS-CKP,54471130
2,T-MON-CKP,52969231
3,ESTADO-SDV-P,52271211
4,QGL,52019710
5,P-ANULAR,51123199
6,P-JUS-CKGL,44689210
7,T-JUS-CKP,16031298
8,T-TPT,10258620
9,P-PDG,7913104


### Validating the Processed Dataset Structure

The processed dataset is checked against the original dataset structure.

The validation confirms that:

- all 2,228 files/instances were processed,
- each processed file contains the expected 12 sensor features and 2 label columns,
- timestamps remain the index,
- the number of observations in each file is unchanged, and
- file boundaries remain preserved.

In [53]:
# Recreate the main project and dataset paths.

from pathlib import Path
import pandas as pd

project_path = Path.cwd().parent

dataset_path = (
    project_path
    / "data"
    / "raw"
    / "3W"
    / "dataset"
)

processed_path = (
    project_path
    / "data"
    / "processed"
    / "3W"
)


# Recreate the final columns used in the processed dataset.

final_columns = (
    final_selected_sensors
    + ["class", "state"]
)


# Find all processed Parquet files.

processed_files = sorted(
    processed_path.glob("*/*.parquet")
)


# Validate the structure of the processed dataset.

validation_results = []

for file_path in processed_files:

    # Read the processed file.
    processed_recording = pd.read_parquet(
        file_path,
        columns=final_columns
    )

    # Locate the corresponding original file.
    original_file = (
        dataset_path
        / file_path.parent.name
        / file_path.name
    )

    # Read the original file for comparison.
    original_recording = pd.read_parquet(
        original_file,
        columns=final_columns
    )

    # Store validation results for this file.
    validation_results.append(
        {
            "recording_id": file_path.stem,
            "observations_original": len(
                original_recording
            ),
            "observations_processed": len(
                processed_recording
            ),
            "columns_correct": (
                list(processed_recording.columns)
                == final_columns
            ),
            "timestamp_index": (
                processed_recording.index.equals(
                    original_recording.index
                )
            )
        }
    )


# Convert the validation results into a DataFrame.

validation_df = pd.DataFrame(
    validation_results
)


# Display the validation results.

print(
    "Processed files:",
    len(validation_df)
)

print(
    "Files with changed observation count:",
    (
        validation_df["observations_original"]
        != validation_df["observations_processed"]
    ).sum()
)

print(
    "Files with incorrect columns:",
    (~validation_df["columns_correct"]).sum()
)

print(
    "Files with changed timestamps:",
    (~validation_df["timestamp_index"]).sum()
)

Processed files: 2228
Files with changed observation count: 0
Files with incorrect columns: 0
Files with changed timestamps: 0


### Final Preprocessing Validation

The processed dataset has passed the structural validation checks.

The final validation confirms that:

- all 2,228 files/instances were processed,
- observation counts are unchanged,
- the expected sensor and label columns are present,
- timestamps are unchanged, and
- short internal gaps were interpolated only for continuous sensors.

The remaining missing values represent longer gaps or periods of sensor unavailability and are intentionally preserved.

In [54]:
# Create a final validation summary.

total_processed_files = len(validation_df)

changed_observation_counts = (
    validation_df["observations_original"]
    != validation_df["observations_processed"]
).sum()

incorrect_columns = (
    ~validation_df["columns_correct"]
).sum()

changed_timestamps = (
    ~validation_df["timestamp_index"]
).sum()

total_interpolated_values = (
    interpolation_summary_df["values_interpolated"]
    .sum()
)

print("Final preprocessing validation")
print("--------------------------------")
print("Processed files:", total_processed_files)
print(
    "Files with changed observation count:",
    changed_observation_counts
)
print(
    "Files with incorrect columns:",
    incorrect_columns
)
print(
    "Files with changed timestamps:",
    changed_timestamps
)
print(
    "Total interpolated observations:",
    total_interpolated_values
)

Final preprocessing validation
--------------------------------
Processed files: 2228
Files with changed observation count: 0
Files with incorrect columns: 0
Files with changed timestamps: 0
Total interpolated observations: 23778


### Preprocessing Summary

The 3W dataset was preprocessed while preserving the original file/instance boundaries and timestamp structure.

Four globally unavailable sensors were excluded, and sensor coverage was evaluated using both file-level and observation-level criteria. Twelve sensors were retained as model features, consisting of 11 continuous sensors and 1 status/discrete sensor.

Short internal gaps in the continuous sensors were handled using linear interpolation, resulting in 23,778 interpolated observations. Longer missing periods, boundary gaps, and missing values in the status sensor were preserved.

All 2,228 files/instances were successfully processed. Observation counts, columns, timestamps, and file boundaries remained unchanged.

The resulting processed dataset provides the basis for feature engineering while retaining the temporal structure and sensor availability characteristics of the original data.

In [55]:
# Display the final preprocessing summary.

print("Preprocessing completed successfully.")
print("-------------------------------------")
print("Original files:", 2228)
print("Processed files:", total_processed_files)
print("Selected sensors:", len(final_selected_sensors))
print("Interpolated observations:", total_interpolated_values)
print("Changed observation counts:", changed_observation_counts)
print("Incorrect column structures:", incorrect_columns)
print("Changed timestamps:", changed_timestamps)

Preprocessing completed successfully.
-------------------------------------
Original files: 2228
Processed files: 2228
Selected sensors: 12
Interpolated observations: 23778
Changed observation counts: 0
Incorrect column structures: 0
Changed timestamps: 0
